# A Simple Iterative Algorithm Optimization System

This notebook runs a loop that calls an LLM once each loop iteration. The LLM is given an implementation of an algorithm and asked to make it faster without changing its output. Each candidate improvement that the LLM produces is tested against the original implementation on random inputs to verify that the two outputs agree. The proposed candidate implementation is timed against the current best implementation. If the candidate is correct and faster, it becomes the new implementation for the LLM to improve.

**If this is your first time using a Jupyter notebook:**
To run a cell, bring it to focus (e.g. by clicking on it so that it highlights) and then press **Shift + Enter**. Run each cell in order, as some cells further down depend on the actions of earlier cells.

## **LLM Set-Up**

In [ ]:
%pip install -q openai

Set the provider's base URL and then paste your API key when prompted. The key is hidden and is not saved in the notebook. (For Gemini, use `https://generativelanguage.googleapis.com/v1beta/openai/` as the base URL.)

In [ ]:
from getpass import getpass
from openai import OpenAI

base_url = "PASTE_BASE_URL_HERE"
api_key = getpass("API key: ")

client = OpenAI(base_url=base_url, api_key=api_key)

List the available models, then copy one name into `MODEL` below.

In [ ]:
for model in client.models.list():
    print(model.id)

In [ ]:
MODEL = "PASTE_MODEL_HERE"

def ask(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt}, # The system prompt tells the LLM how it should behave or act
            {"role": "user", "content": user_prompt}, # The user prompt is the query that a user asks the LLM
        ],
    )
    return response.choices[0].message.content

## **Starting Algorithm**

For this example, we'll use the algorithm for factoring an integer by trial division. The code is stored as a string.

In [ ]:
BASELINE_CODE = """
def factor(n):
    # Return the prime factors of n >= 2 in increasing order, counted with multiplicity.
    factors = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)
    return factors
"""

These are helper functions that will be used in our evaluations.
- `load` runs a code string and returns the `factor` function it defines.
- `extract_code` pulls the code out of the LLM's reply.
- `run` applies a function to a list of inputs and reports the fastest of a few timed repetitions, which reduces timing noise.

Note that `load` executes whatever code the LLM writes.

In [ ]:
import random, re, time

def load(code):
    namespace = {}
    exec(code, namespace)
    return namespace["factor"]

def extract_code(reply):
    match = re.search(r"```(?:python)?\n(.*?)```", reply, re.DOTALL)
    return match.group(1) if match else reply

def run(f, inputs, repeats=3):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        outputs = [f(n) for n in inputs]
        times.append(time.perf_counter() - start)
    return outputs, min(times)

print(load(BASELINE_CODE)(360))

Random inputs to be used in each round:

In [ ]:
NUM_INPUTS = 20

def random_inputs():
    return [random.randint(10**11, 10**12) for _ in range(NUM_INPUTS)]

## **Agent Prompt**

The optimizer's instructions:

In [ ]:
OPTIMIZER_PROMPT = """You are an expert in algorithms and Python performance. You are given a Python function factor(n).
Rewrite it so that it runs faster while returning exactly the same output for every integer n >= 2:
a list of the prime factors of n in increasing order, with multiplicity.
If you are shown a previous attempt and its result, use the result to revise or improve your work.
Use only the Python standard library. Reply with the complete code in a single ```python block."""

## **Optimization Iteration**

In each round the LLM proposes a candidate. The candidate is rejected if it raises an error or disagrees with the original implementation on any of the randomly tested inputs. Otherwise it replaces the current best implementation if it is at least `MIN_SPEEDUP` times as fast on those inputs. (The margin keeps timing noise from accepting a candidate that is not actually faster.) If a candidate runs for too long (for example, because of an infinite loop), you may need to interrupt the kernel.

In [ ]:
MAX_ROUNDS = 5
MIN_SPEEDUP = 1.1

reference = load(BASELINE_CODE)
best_code, candidate_code, result = BASELINE_CODE, "", ""

for i in range(1, MAX_ROUNDS + 1):
    request = f"Current implementation:\n```python\n{best_code}\n```"
    if result:
        request += f"\n\nYour previous attempt:\n```python\n{candidate_code}\n```\n\nResult:\n{result}"

    candidate_code = extract_code(ask(OPTIMIZER_PROMPT, request))

    inputs = random_inputs()
    expected, reference_time = run(reference, inputs)
    _, best_time = run(load(best_code), inputs)

    try:
        outputs, candidate_time = run(load(candidate_code), inputs)
    except Exception as e:
        result = f"REJECTED: the code raised an error: {e!r}"
    else:
        wrong = [(n, a, b) for n, a, b in zip(inputs, outputs, expected) if a != b]
        if wrong:
            n, a, b = wrong[0]
            result = f"REJECTED: factor({n}) returned {a}, but the correct output is {b}."
        elif candidate_time * MIN_SPEEDUP < best_time:
            best_code = candidate_code
            result = f"ACCEPTED: {candidate_time:.4f}s versus {best_time:.4f}s for the current implementation."
        else:
            result = f"REJECTED: correct, but {candidate_time:.4f}s is not sufficiently faster than {best_time:.4f}s for the current implementation."

    print(f"===== Round {i} =====\n\nCANDIDATE:\n{candidate_code}\n\nRESULT:\n{result}\n(original implementation: {reference_time:.4f}s)\n")

print("Final implementation:\n")
print(best_code)